# 🌿 Kalmegh – Plant Disease Detection
### Training Notebook: Feature Extraction + ML Models + CNN + Graphs

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import joblib

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from feature_extractor import extract_features, preprocess_image

print('Libraries loaded successfully')

## 1. Visualise Sample Images

In [ ]:
DATA_DIR = '../data/train'
LABELS   = {'healthy': 0, 'diseased': 1}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, (label, _) in enumerate(LABELS.items()):
    folder = os.path.join(DATA_DIR, label)
    images = glob.glob(os.path.join(folder, '*.jpg'))[:4]
    for col, img_path in enumerate(images):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        axes[row][col].set_title(label.capitalize())
        axes[row][col].axis('off')
plt.suptitle('Sample Leaf Images', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Preprocessing Pipeline Visualisation

In [ ]:
sample_images = glob.glob(os.path.join(DATA_DIR, 'healthy', '*.jpg'))[:1]
if sample_images:
    raw = cv2.imread(sample_images[0])
    raw_rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    blurred = cv2.GaussianBlur(raw, (5,5), 0)
    resized = cv2.resize(blurred, (128, 128))
    normalized = resized.astype(np.float32) / 255.0

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_rgb); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)); axes[1].set_title('Denoised'); axes[1].axis('off')
    axes[2].imshow(normalized); axes[2].set_title('Resized & Normalized'); axes[2].axis('off')
    plt.suptitle('Preprocessing Steps')
    plt.tight_layout()
    plt.show()
else:
    print('Add images to data/train/healthy to see preprocessing.')

## 3. Load Dataset & Extract Features

In [ ]:
X, y = [], []
for label_name, label_id in LABELS.items():
    folder = os.path.join(DATA_DIR, label_name)
    images = glob.glob(os.path.join(folder, '*.jpg')) + \
             glob.glob(os.path.join(folder, '*.png'))
    print(f'[{label_name}] {len(images)} images')
    for img_path in images:
        try:
            X.append(extract_features(img_path))
            y.append(label_id)
        except Exception as e:
            print(f'  Skip {img_path}: {e}')

X, y = np.array(X), np.array(y)

# Use synthetic data if no images found
if len(X) < 10:
    print('\nUsing synthetic demo data (add real images to data/train/)')
    np.random.seed(42)
    X = np.random.rand(100, 217)
    y = np.array([0]*50 + [1]*50)

print(f'\nDataset shape: {X.shape}, Labels: {np.bincount(y)}')

## 4. Class Distribution

In [ ]:
counts = np.bincount(y)
plt.figure(figsize=(5, 4))
plt.bar(['Healthy', 'Diseased'], counts, color=['#66bb6a', '#ef5350'])
plt.title('Class Distribution')
plt.ylabel('Count')
for i, c in enumerate(counts):
    plt.text(i, c + 0.5, str(c), ha='center')
plt.tight_layout()
plt.show()

## 5. Train & Evaluate ML Models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'SVM':           SVC(kernel='rbf', C=10, gamma='scale', probability=True),
    'KNN':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    cv  = cross_val_score(model, X_train_s, y_train, cv=5)
    results[name] = {'accuracy': acc, 'cv_mean': cv.mean(), 'cv_std': cv.std(), 'y_pred': y_pred}
    print(f'{name}: Test={acc*100:.1f}%  CV={cv.mean()*100:.1f}%±{cv.std()*100:.1f}%')

## 6. Accuracy Comparison Chart

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy']*100 for n in names]
cv_means = [results[n]['cv_mean']*100 for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, accs, width, label='Test Accuracy', color='#4C72B0')
bars2 = ax.bar(x + width/2, cv_means, width, label='CV Accuracy', color='#DD8452')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 115)
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Healthy','Diseased'],
                yticklabels=['Healthy','Diseased'])
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

## 8. CNN Training (Bonus)

In [ ]:
# Run CNN training from the parent directory
# Uncomment when you have enough images (>50 per class)

# import subprocess
# subprocess.run(['python', '../train_cnn.py'])

# Or load and display saved CNN curves:
cnn_curve_path = '../models/cnn_training_curves.png'
if os.path.exists(cnn_curve_path):
    from PIL import Image
    img = Image.open(cnn_curve_path)
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title('CNN Training Curves')
    plt.show()
else:
    print('Run train_cnn.py first to generate CNN curves.')

## 9. Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = models[best_name]
safe = best_name.lower().replace(' ', '_')
joblib.dump(best_model, f'../models/{safe}.pkl')

with open('../models/best_model.txt', 'w') as f:
    f.write(safe)

print(f'Best model saved: {best_name} ({results[best_name]["accuracy"]*100:.1f}%)')